In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
packageVersion("marginaleffects")

[1] ‘0.25.1’

In [4]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "GraduationPaper/RevisetoJournal/9991-MergedData_similarity.csv"))
dim(data)

[1] 317275    103

In [5]:
print(names(data))

  [1] "X"                                           
  [2] "work_id"                                     
  [3] "PublishedYear"                               
  [4] "Facility"                                    
  [5] "num_fac"                                     
  [6] "paper_type"                                  
  [7] "paper_language"                              
  [8] "novel_uzzi"                                  
  [9] "novel_uzzi_bin"                              
 [10] "num_fac_scientist"                           
 [11] "ratio_fac_scientist"                         
 [12] "bin_fac_scientist"                           
 [13] "text_fac_scientist"                          
 [14] "fac_scientist_team"                          
 [15] "num_leader"                                  
 [16] "ratio_leader"                                
 [17] "bin_leader"                                  
 [18] "fac_scientist_lead_relratio"                 
 [19] "fac_scientist_lead_num"                

In [6]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                               PublishedYear 
                                           0 
                                    Facility 
                                           0 
                                     num_fac 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                  novel_uzzi 
                                        2532 
                              novel_uzzi_bin 
                                           0 
                           num_fac_scientist 
                                           0 
                         ratio_fac_scientist 
                                           0 
                           bin_fac_scientist 
                                           0 
                          text_fac_scientist 
                                           0 
                          fac_scientist_team 
                                           0 
                                  num_leader 
                                           0 
                                ratio_leader 
                                           0 
                                  bin_leader 
                                           0 
                 fac_scientist_lead_relratio 
                                           0 
                      fac_scientist_lead_num 
                                           0 
                    fac_scientist_lead_ratio 
                                           0 
                      fac_scientist_lead_bin 
                                           0 
                     fac_scientist_lead_text 
                                           0 
                                      CoType 
                                           0 
                        CoType_Collaboration 
                                           0 
                        CoType_Participation 
                                           0 
                              CoType_Service 
                                           0 
                                lnnum_author 
                                           0 
                                  lnnum_inst 
                                           0 
                               lnnum_country 
                                           0 
                               international 
                                           0 
                             lnnum_reference 
                                           0 
                                 open_access 
                                           0 
                                 RaoStirling 
                                           0 
                                         SDG 
                                           0 
                               lntimescited5 
                                       14695 
                              lntimescited10 
                                       12880 
                             lntimescitedall 
                                           0 
                                 lnab_length 
                                           0 
                           lnmean_career_age 
                                           0 
                       lnex_ld_avg_avgimpact 
                                           0 
                      lnex_ld_avg_insthindex 
                                           0 
                                ex_ld_bin_gs 
                                           0 
                              ex_ld_ratio_gs 
                                           0 
                             ex_ld_bin_sameC 
                                         

In [7]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [8]:
# data <- data %>% filter(!is.na(mean_career_age))
# data <- data %>% filter(!is.na(frac_hype_words))
# data <- data %>% filter(!is.na(source_hindex))
# data <- data %>% filter(!is.na(open_access))
# dim(data)

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$Facility <- as.factor(data$Facility)

In [11]:
data$CoType <- factor(data$CoType)
data <- within(data, CoType <- relevel(CoType, ref = 'Service'))
data$paper_type <- factor(data$paper_type)
data <- within(data, paper_type <- relevel(paper_type, ref = 'review'))
data$text_fac_scientist <- factor(data$text_fac_scientist)
data <- within(data, text_fac_scientist <- relevel(text_fac_scientist, ref = 'NonStaffPart'))
data$fac_scientist_lead_text <- factor(data$fac_scientist_lead_text)
data <- within(data, fac_scientist_lead_text <- relevel(fac_scientist_lead_text, ref = 'NonStaffLead'))
data$open_access <- factor(data$open_access)
data <- within(data, open_access <- relevel(open_access, ref = 'False'))
data$SDG <- factor(data$SDG)
data <- within(data, SDG <- relevel(SDG, ref = 'False'))
data$ex_ld_bin_sameC <- factor(data$ex_ld_bin_sameC)
data <- within(data, ex_ld_bin_sameC <- relevel(ex_ld_bin_sameC, ref = 'NonSame'))
data$ex_ld_bin_gs <- factor(data$ex_ld_bin_gs)
data <- within(data, ex_ld_bin_gs <- relevel(ex_ld_bin_gs, ref = 'GlobalSouth'))
data$ex_ld_max_before_year_with_ih_bin <- factor(data$ex_ld_max_before_year_with_ih_bin)
data <- within(data, ex_ld_max_before_year_with_ih_bin <- relevel(ex_ld_max_before_year_with_ih_bin, ref = 'False'))
data$ex_ld_max_before_year_participation_bin <- factor(data$ex_ld_max_before_year_participation_bin)
data <- within(data, ex_ld_max_before_year_participation_bin <- relevel(ex_ld_max_before_year_participation_bin, ref = 'False'))
data$ex_ld_max_before_year_co_lead_bin <- factor(data$ex_ld_max_before_year_co_lead_bin)
data <- within(data, ex_ld_max_before_year_co_lead_bin <- relevel(ex_ld_max_before_year_co_lead_bin, ref = 'False'))
data$international <- factor(data$international)
data <- within(data, international <- relevel(international, ref = 'domestic'))

In [12]:
paper_level <- "lnnum_author + international + lnnum_reference + num_fac + SDG + lnmean_career_age"
ex_controls <- "lnex_ld_avg_avgimpact + lnex_ld_avg_insthindex + ex_ld_bin_gs + ex_ld_bin_sameC + knowledge_proximity_mean"
moderating <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_with_ih_bin"
moderating2 <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_participation_bin"
moderating3 <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_co_lead_bin"
disciplines <- "Agricultural.and.Biological.Sciences + Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [13]:
paper_vars <- c("lnnum_author", "international", "lnnum_reference", "num_fac", "SDG", "lnmean_career_age")
ex_vars <- c("lnex_ld_avg_avgimpact", "lnex_ld_avg_insthindex", "ex_ld_bin_gs", "ex_ld_bin_sameC", "knowledge_proximity_mean")
moderating_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_with_ih_bin")
moderating2_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_participation_bin")
moderating3_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_co_lead_bin")
disciplines_vars <- c("Agricultural.and.Biological.Sciences", "Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

# H1:With > Without

In [14]:
fml <- as.formula(
  paste0("novel_uzzi ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_total_bin <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], vcov = "hetero")
summary(model_total_bin)

NOTE: 98 observations removed because of NA values (LHS: 98).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 294,781
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    t value
text_fac_scientistStaffPart                   -7.47791   0.646703 -11.563136
lnnum_author                                  21.57048   0.754618  28.584639
internationalinternational                     3.20532   0.674055   4.755278
lnnum_reference                              -17.97535   1.211354 -14.839064
num_fac                                      -11.74277   0.290126 -40.474715
SDGTrue                                       -1.53994   0.553439  -2.782501
lnmean_career_age                              1.49432   1.023511   1.459999
lnex_ld_avg_avgimpact                          7.79411   0.488205  15.964826
lnex_ld_avg_insthindex                         2.45220   0.509605   4.811975
ex_ld_bin_gsGlobalNorth                       -9.22934   1.175214  -7.853324
ex_ld_bin_sameCS

In [15]:
# 每组 reg_class 的平均预测概率
pred_bin <- avg_predictions(model_total_bin, variables = "text_fac_scientist")
pred_bin

text_fac_scientist,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NonStaffPart,36.44608,11.01445,3.308932,0.0009365265,10.060393,14.858144,58.03401
StaffPart,28.96816,10.88206,2.662012,0.0077675105,7.008332,7.639726,50.29660


In [16]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_pred.csv")
# write.csv(pred_bin, fname, row.names = FALSE)

In [17]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & -7.48$^{***}$\\   
                                                       & (0.647)\\   
   lnnum\_author                                       & 21.6$^{***}$\\   
                                                       & (0.755)\\   
   internationalinternational                          & 3.21$^{***}$\\   
                                                       & (0.674)\\   
   lnnum\_reference                                    & -18.0$^{***}$\\   
                                                       & (1.21)\\   
   num\_fac                                            & -11.7$^{***}$\\   
                                                       & (0.290)\\   
   SDGTrue          

In [18]:
margins_eff_bin <- avg_comparisons(model_total_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),0.7948226,0.06039282,13.16088,1.473609e-39,128.9958,0.6764548,0.9131903,27.95618,20.47826,27.95618


In [19]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_comp_ratio.csv")
# write.csv(margins_eff_bin, fname, row.names = FALSE)

# H1 different disciplines

In [20]:
fml <- as.formula(
  paste0("novel_uzzi ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps_bin <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], vcov = "hetero")
summary(model_ps_bin)

NOTE: 92 observations removed because of NA values (LHS: 92).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 253,483
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    t value
text_fac_scientistStaffPart                   -8.465628   0.706548 -11.981669
lnnum_author                                  23.984848   0.819365  29.272481
internationalinternational                     4.053309   0.785982   5.156998
lnnum_reference                              -19.358400   1.338243 -14.465534
num_fac                                      -13.397625   0.331024 -40.473233
SDGTrue                                       -0.611081   0.638804  -0.956603
lnmean_career_age                              2.305741   1.179369   1.955063
lnex_ld_avg_avgimpact                         10.930593   0.612292  17.851924
lnex_ld_avg_insthindex                         2.045770   0.590034   3.467207
ex_ld_bin_gsGlobalNorth                       -9.069383   1.289815  -7.031538
ex_ld

In [21]:
# margins_eff_ps_bin <- avg_comparisons(model_ps_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_ps_bin

In [22]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ps_comp_ratio.csv")
# write.csv(margins_eff_ps_bin, fname, row.names = FALSE)

In [23]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & -8.47$^{***}$\\   
                                                       & (0.707)\\   
   lnnum\_author                                       & 24.0$^{***}$\\   
                                                       & (0.819)\\   
   internationalinternational                          & 4.05$^{***}$\\   
                                                       & (0.786)\\   
   lnnum\_reference                                    & -19.4$^{***}$\\   
                                                       & (1.34)\\   
   num\_fac                                            & -13.4$^{***}$\\   
                                                       & (0.331)\\   
   SDGTrue          

In [24]:
fml <- as.formula(
  paste0("novel_uzzi ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ls_bin <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], vcov = "hetero")
summary(model_ls_bin)

NOTE: 12 observations removed because of NA values (LHS: 12).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 77,579
Fixed-effects: PublishedYear: 49
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    t value
text_fac_scientistStaffPart                   -1.207487   0.485872  -2.485193
lnnum_author                                  -5.592646   0.432067 -12.943927
internationalinternational                    -1.217497   0.318326  -3.824680
lnnum_reference                               -9.145027   0.854868 -10.697584
num_fac                                        0.351628   0.200430   1.754366
SDGTrue                                       -3.052270   0.343181  -8.894048
lnmean_career_age                             -2.231310   0.585706  -3.809611
lnex_ld_avg_avgimpact                          6.248172   0.351179  17.792003
lnex_ld_avg_insthindex                         3.268469   0.379414   8.614525
ex_ld_bin_gsGlobalNorth                       -1.220556   0.619215  -1.971135
ex_ld_

In [25]:
# margins_eff_ls_bin <- avg_comparisons(model_ls_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_ls_bin

In [26]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ls_comp_ratio.csv")
# write.csv(margins_eff_ls_bin, fname, row.names = FALSE)

In [27]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & -1.21$^{**}$\\   
                                                       & (0.486)\\   
   lnnum\_author                                       & -5.59$^{***}$\\   
                                                       & (0.432)\\   
   internationalinternational                          & -1.22$^{***}$\\   
                                                       & (0.318)\\   
   lnnum\_reference                                    & -9.15$^{***}$\\   
                                                       & (0.855)\\   
   num\_fac                                            & 0.352$^{*}$\\   
                                                       & (0.200)\\   
   SDGTrue          

In [28]:
fml <- as.formula(
  paste0("novel_uzzi ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_hs_bin <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], vcov = "hetero")
summary(model_hs_bin)

NOTE: 8 observations removed because of NA values (LHS: 8).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 33,409
Fixed-effects: PublishedYear: 46
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error   t value
text_fac_scientistStaffPart                   -6.163688   1.545622 -3.987837
lnnum_author                                  -2.578085   1.101526 -2.340467
internationalinternational                     1.195446   0.917227  1.303327
lnnum_reference                              -12.413968   2.239326 -5.543617
num_fac                                       -0.185456   0.358998 -0.516595
SDGTrue                                       -2.301709   0.831400 -2.768474
lnmean_career_age                             -1.122964   1.074451 -1.045151
lnex_ld_avg_avgimpact                          2.417556   0.653871  3.697299
lnex_ld_avg_insthindex                         2.818443   0.724040  3.892665
ex_ld_bin_gsGlobalNorth                       -2.991551   3.201607 -0.934391
ex_ld_bin_sameCSa

In [29]:
# margins_eff_hs_bin <- avg_comparisons(model_hs_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_hs_bin

In [30]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_hs_comp_ratio.csv")
# write.csv(margins_eff_hs_bin, fname, row.names = FALSE)

In [31]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & -6.16$^{***}$\\   
                                                       & (1.55)\\   
   lnnum\_author                                       & -2.58$^{**}$\\   
                                                       & (1.10)\\   
   internationalinternational                          & 1.20\\   
                                                       & (0.917)\\   
   lnnum\_reference                                    & -12.4$^{***}$\\   
                                                       & (2.24)\\   
   num\_fac                                            & -0.185\\   
                                                       & (0.359)\\   
   SDGTrue                           

In [32]:
fml <- as.formula(
  paste0("novel_uzzi ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_nps_bin <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], vcov = "hetero")
summary(model_nps_bin)

NOTE: 6 observations removed because of NA values (LHS: 6).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 41,298
Fixed-effects: PublishedYear: 44
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    t value
text_fac_scientistStaffPart                   -3.136065   0.929138  -3.375242
lnnum_author                                  -6.765645   0.521514 -12.973085
internationalinternational                    -1.483764   0.396163  -3.745335
lnnum_reference                               -6.393325   1.687296  -3.789096
num_fac                                        0.858191   0.271677   3.158867
SDGTrue                                       -4.757647   0.410928 -11.577809
lnmean_career_age                             -4.865555   0.611893  -7.951643
lnex_ld_avg_avgimpact                          7.330653   0.339054  21.620878
lnex_ld_avg_insthindex                         3.181002   0.322193   9.872964
ex_ld_bin_gsGlobalNorth                       -0.969013   0.809865  -1.196512
ex_ld_

In [33]:
# margins_eff_nps_bin <- avg_comparisons(model_nps_bin, variables = "text_fac_scientist", comparison = 'ratio')
# margins_eff_nps_bin

In [34]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_nps_comp_ratio.csv")
# write.csv(margins_eff_nps_bin, fname, row.names = FALSE)

In [35]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & -3.14$^{***}$\\   
                                                       & (0.929)\\   
   lnnum\_author                                       & -6.77$^{***}$\\   
                                                       & (0.522)\\   
   internationalinternational                          & -1.48$^{***}$\\   
                                                       & (0.396)\\   
   lnnum\_reference                                    & -6.39$^{***}$\\   
                                                       & (1.69)\\   
   num\_fac                                            & 0.858$^{***}$\\   
                                                       & (0.272)\\   
   SDGTrue        

# H2: Collaboration > Participation

In [36]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_total <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], vcov = "hetero")
summary(model_total)

NOTE: 98 observations removed because of NA values (LHS: 98).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 294,781
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    t value
CoTypeCollaboration                           -9.96637   0.652299 -15.278836
CoTypeParticipation                           -2.15681   1.080937  -1.995312
lnnum_author                                  21.07243   0.754953  27.912245
internationalinternational                     3.25540   0.672961   4.837428
lnnum_reference                              -18.02953   1.213173 -14.861469
num_fac                                      -11.69435   0.289572 -40.384993
SDGTrue                                       -1.49003   0.552596  -2.696418
lnmean_career_age                              1.71602   1.027390   1.670271
lnex_ld_avg_avgimpact                          7.87966   0.488332  16.135854
lnex_ld_avg_insthindex                         2.42266   0.509157   4.758176
ex_ld_bin_gsGlob

In [37]:
# # 每组 reg_class 的平均预测概率
# pred <- avg_predictions(model_total, variables = "CoType")
# pred

In [38]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_pred.csv")
# write.csv(pred, fname, row.names = FALSE)

In [39]:
# # 每组 reg_class 的平均预测概率
# margins_eff <- avg_comparisons(model_total, variables = "CoType", comparison = 'ratio')
# margins_eff

In [40]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_comp_ratio.csv")
# write.csv(margins_eff, fname, row.names = FALSE)

In [41]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & -9.97$^{***}$\\   
                                                       & (0.652)\\   
   CoTypeParticipation                                 & -2.16$^{**}$\\   
                                                       & (1.08)\\   
   lnnum\_author                                       & 21.1$^{***}$\\   
                                                       & (0.755)\\   
   internationalinternational                          & 3.26$^{***}$\\   
                                                       & (0.673)\\   
   lnnum\_reference                                    & -18.0$^{***}$\\   
                                                       & (1.21)\\   
   num\_fac           

# H2 Discipline

In [42]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], vcov = "hetero")
summary(model_ps)

NOTE: 92 observations removed because of NA values (LHS: 92).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 253,483
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    t value
CoTypeCollaboration                          -10.680946   0.708625 -15.072769
CoTypeParticipation                           -3.690634   1.166319  -3.164343
lnnum_author                                  23.491284   0.822742  28.552427
internationalinternational                     4.098818   0.784869   5.222298
lnnum_reference                              -19.404254   1.340109 -14.479610
num_fac                                      -13.347596   0.330446 -40.392715
SDGTrue                                       -0.572494   0.638011  -0.897310
lnmean_career_age                              2.510982   1.184017   2.120732
lnex_ld_avg_avgimpact                         11.025679   0.612878  17.989999
lnex_ld_avg_insthindex                         2.019775   0.589581   3.425778
ex_ld

In [43]:
# # 每组 reg_class 的平均预测概率
# margins_eff_ps <- avg_comparisons(model_ps, variables = "CoType", comparison = 'ratio')
# margins_eff_ps

In [44]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ps_comp_ratio.csv")
# write.csv(margins_eff_ps, fname, row.names = FALSE)

In [45]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & -10.7$^{***}$\\   
                                                       & (0.709)\\   
   CoTypeParticipation                                 & -3.69$^{***}$\\   
                                                       & (1.17)\\   
   lnnum\_author                                       & 23.5$^{***}$\\   
                                                       & (0.823)\\   
   internationalinternational                          & 4.10$^{***}$\\   
                                                       & (0.785)\\   
   lnnum\_reference                                    & -19.4$^{***}$\\   
                                                       & (1.34)\\   
   num\_fac          

In [46]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ls <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], vcov = "hetero")
summary(model_ls)

NOTE: 12 observations removed because of NA values (LHS: 12).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 77,579
Fixed-effects: PublishedYear: 49
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    t value
CoTypeCollaboration                           -1.427616   0.563684  -2.532653
CoTypeParticipation                           -0.651302   0.718016  -0.907085
lnnum_author                                  -5.595249   0.431546 -12.965592
internationalinternational                    -1.220636   0.318983  -3.826650
lnnum_reference                               -9.147517   0.854968 -10.699244
num_fac                                        0.354596   0.200630   1.767411
SDGTrue                                       -3.050865   0.343021  -8.894091
lnmean_career_age                             -2.214193   0.586946  -3.772396
lnex_ld_avg_avgimpact                          6.250787   0.350737  17.821880
lnex_ld_avg_insthindex                         3.270029   0.379366   8.619712
ex_ld_

In [47]:
# # 每组 reg_class 的平均预测概率
# margins_eff_ls <- avg_comparisons(model_ls, variables = "CoType", comparison = 'ratio')
# margins_eff_ls

In [48]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ls_comp_ratio.csv")
# write.csv(margins_eff_ls, fname, row.names = FALSE)

In [49]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & -1.43$^{**}$\\   
                                                       & (0.564)\\   
   CoTypeParticipation                                 & -0.651\\   
                                                       & (0.718)\\   
   lnnum\_author                                       & -5.60$^{***}$\\   
                                                       & (0.432)\\   
   internationalinternational                          & -1.22$^{***}$\\   
                                                       & (0.319)\\   
   lnnum\_reference                                    & -9.15$^{***}$\\   
                                                       & (0.855)\\   
   num\_fac              

In [50]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_hs <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], vcov = "hetero")
summary(model_hs)

NOTE: 8 observations removed because of NA values (LHS: 8).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 33,409
Fixed-effects: PublishedYear: 46
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error   t value
CoTypeCollaboration                           -6.989213   1.852211 -3.773443
CoTypeParticipation                           -4.427218   1.995193 -2.218943
lnnum_author                                  -2.612084   1.100981 -2.372506
internationalinternational                     1.182021   0.919404  1.285639
lnnum_reference                              -12.416814   2.238838 -5.546097
num_fac                                       -0.187018   0.359222 -0.520620
SDGTrue                                       -2.292495   0.829648 -2.763214
lnmean_career_age                             -1.072274   1.068849 -1.003204
lnex_ld_avg_avgimpact                          2.426588   0.656437  3.696607
lnex_ld_avg_insthindex                         2.822234   0.723453  3.901057
ex_ld_bin_gsGloba

In [51]:
# # 每组 reg_class 的平均预测概率
# margins_eff_hs <- avg_comparisons(model_hs, variables = "CoType", comparison = 'ratio')
# margins_eff_hs

In [52]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_hs_comp_ratio.csv")
# write.csv(margins_eff_hs, fname, row.names = FALSE)

In [53]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & -6.99$^{***}$\\   
                                                       & (1.85)\\   
   CoTypeParticipation                                 & -4.43$^{**}$\\   
                                                       & (2.00)\\   
   lnnum\_author                                       & -2.61$^{**}$\\   
                                                       & (1.10)\\   
   internationalinternational                          & 1.18\\   
                                                       & (0.919)\\   
   lnnum\_reference                                    & -12.4$^{***}$\\   
                                                       & (2.24)\\   
   num\_fac                     

In [54]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_nps <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], vcov = "hetero")
summary(model_nps)

NOTE: 6 observations removed because of NA values (LHS: 6).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 41,298
Fixed-effects: PublishedYear: 44
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    t value
CoTypeCollaboration                           -3.414270   0.798505  -4.275830
CoTypeParticipation                           -2.548076   1.849937  -1.377385
lnnum_author                                  -6.769396   0.518203 -13.063219
internationalinternational                    -1.484360   0.396095  -3.747482
lnnum_reference                               -6.395976   1.684931  -3.795986
num_fac                                        0.860175   0.271122   3.172648
SDGTrue                                       -4.754423   0.413033 -11.511009
lnmean_career_age                             -4.850856   0.617512  -7.855482
lnex_ld_avg_avgimpact                          7.331620   0.339462  21.597749
lnex_ld_avg_insthindex                         3.183849   0.321926   9.890008
ex_ld_

In [55]:
# # 每组 reg_class 的平均预测概率
# margins_eff_nps <- avg_comparisons(model_nps, variables = "CoType", comparison = 'ratio')
# margins_eff_nps

In [56]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_nps_comp_ratio.csv")
# write.csv(margins_eff_nps, fname, row.names = FALSE)

In [57]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & -3.41$^{***}$\\   
                                                       & (0.799)\\   
   CoTypeParticipation                                 & -2.55\\   
                                                       & (1.85)\\   
   lnnum\_author                                       & -6.77$^{***}$\\   
                                                       & (0.518)\\   
   internationalinternational                          & -1.48$^{***}$\\   
                                                       & (0.396)\\   
   lnnum\_reference                                    & -6.40$^{***}$\\   
                                                       & (1.68)\\   
   num\_fac                

# H3: Too much will suppress

# H3a: Participation too much not good

In [58]:
fml <- as.formula(
  paste0("novel_uzzi ~ ratio_fac_scientist + I(ratio_fac_scientist^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_h3_pratio <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], vcov = "hetero")
summary(model_h3_pratio)

NOTE: 98 observations removed because of NA values (LHS: 98).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 294,781
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    t value
ratio_fac_scientist                          -49.16323   4.693222 -10.475369
I(ratio_fac_scientist^2)                      45.21122   8.755500   5.163751
lnnum_author                                  20.82863   0.754383  27.610170
internationalinternational                     3.41035   0.670711   5.084681
lnnum_reference                              -18.09063   1.211922 -14.927222
num_fac                                      -11.47137   0.288355 -39.782120
SDGTrue                                       -1.51644   0.554286  -2.735848
lnmean_career_age                              1.66885   1.024643   1.628713
lnex_ld_avg_avgimpact                          7.81831   0.489160  15.983125
lnex_ld_avg_insthindex                         2.34477   0.511548   4.583679
ex_ld_bin_gsGlob

In [59]:
# library(marginaleffects)
# # 设置 draw = FALSE，直接拦截绘图数据
# plot_data <- plot_predictions(model_h3_pratio, condition = "ratio_fac_scientist", draw = FALSE)
# # 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
# export_data <- plot_data[, c("ratio_fac_scientist", "estimate", "conf.low", "conf.high")]
# # # 导出为 CSV 文件，给 Python 准备
# write.csv(export_data, "R_ex_ld_h3_pred_pratio.csv", row.names = FALSE)
# export_data
# # print("数据已成功导出！")
# # head(export_data)

In [60]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_pratio,
                           keep = c("ratio_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   ratio\_fac\_scientist                               & -49.2$^{***}$\\   
                                                       & (4.69)\\   
   ratio\_fac\_scientist square                        & 45.2$^{***}$\\   
                                                       & (8.76)\\   
   lnnum\_author                                       & 20.8$^{***}$\\   
                                                       & (0.754)\\   
   internationalinternational                          & 3.41$^{***}$\\   
                                                       & (0.671)\\   
   lnnum\_reference                                    & -18.1$^{***}$\\   
                                                       & (1.21)\\   
   num\_fac            

# H3b: Lead too much not good

In [61]:
fml <- as.formula(
  paste0("novel_uzzi ~ fac_scientist_lead_ratio + I(fac_scientist_lead_ratio^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_h3_lratio <- feols(fml, data = data[(data$paper_type =='article')&(data$CoType_Service==0)&(data$knowledge_proximity_mean>0), ], vcov = "hetero")
summary(model_h3_lratio)

NOTE: 14 observations removed because of NA values (LHS: 14).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 78,519
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    t value
fac_scientist_lead_ratio                      11.489314   6.756742   1.700422
I(fac_scientist_lead_ratio^2)                 -7.083996   9.173371  -0.772235
lnnum_author                                  41.356115   1.081579  38.236802
internationalinternational                    -0.393851   1.183209  -0.332867
lnnum_reference                              -16.324329   2.379446  -6.860560
num_fac                                      -12.198191   0.456036 -26.748301
SDGTrue                                       -2.913139   0.932859  -3.122806
lnmean_career_age                              3.682351   1.495649   2.462041
lnex_ld_avg_avgimpact                          3.576699   0.781519   4.576597
lnex_ld_avg_insthindex                         4.912999   0.799729   6.143332
ex_ld_

In [62]:
# library(marginaleffects)
# # 设置 draw = FALSE，直接拦截绘图数据
# plot_data <- plot_predictions(model_h3_lratio, condition = "fac_scientist_lead_ratio", draw = FALSE)
# # 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
# export_data <- plot_data[, c("fac_scientist_lead_ratio", "estimate", "conf.low", "conf.high")]
# # # 导出为 CSV 文件，给 Python 准备
# write.csv(export_data, "R_ex_ld_h3_pred_lratio.csv", row.names = FALSE)
# export_data
# # print("数据已成功导出！")
# # head(export_data)

In [63]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_lratio,
                           keep = c("fac_scientist_lead_ratio", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\\   
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_scientist\_lead\_ratio                         & 11.5$^{*}$\\   
                                                       & (6.76)\\   
   fac\_scientist\_lead\_ratio square                  & -7.08\\   
                                                       & (9.17)\\   
   lnnum\_author                                       & 41.4$^{***}$\\   
                                                       & (1.08)\\   
   internationalinternational                          & -0.394\\   
                                                       & (1.18)\\   
   lnnum\_reference                                    & -16.3$^{***}$\\   
                                                       & (2.38)\\   
   num\_fac                              

# Moderating

In [64]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType*lnex_ld_avg_before_year_prod_fac  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_facpub <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], vcov = "hetero")
summary(model_pre_facpub)

NOTE: 98 observations removed because of NA values (LHS: 98).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 294,781
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                       Estimate Std. Error
CoTypeCollaboration                                   -5.513123   1.984836
CoTypeParticipation                                  -37.516598   3.174151
lnex_ld_avg_before_year_prod_fac                       0.132528   0.513220
lnnum_author                                          19.454120   0.732072
internationalinternational                             3.699588   0.673092
lnnum_reference                                      -18.060777   1.213206
num_fac                                              -11.703731   0.281608
SDGTrue                                               -1.394094   0.553021
lnmean_career_age                                      1.936293   1.030843
lnex_ld_avg_avgimpact                                  8.031189   0.487130
lnex_ld_avg_insthindex                

In [65]:
# # 1. 找到你这个连续变量的实际最小值和最大值（假设是 0 和 10，你需要改成你的实际极值）
# min_val <- min(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# max_val <- max(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# # 2. 运行估计
# res_pre_facpub <- avg_comparisons(
#     model_pre_facpub,
#     variables = "CoType",
#     comparison = "ratio",
#     newdata = datagrid(
#     model = model_pre_facpub,
#     lnex_ld_avg_before_year_prod_fac = seq(min_val, max_val, length.out = 50) # 这里的 10 可以改成任意你想要的数字
#   ),
#     by = "lnex_ld_avg_before_year_prod_fac"
# )
# res_pre_facpub

In [66]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_facpub.csv")
# write.csv(res_pre_facpub, fname, row.names = FALSE)

In [67]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_facpub,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                        & novel\_uzzi\\   
   Model:                                                                     & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                        & -5.51$^{***}$\\   
                                                                              & (1.98)\\   
   CoTypeParticipation                                                        & -37.5$^{***}$\\   
                                                                              & (3.17)\\   
   lnex\_ld\_avg\_before\_year\_prod\_fac                                     & 0.133\\   
                                                                              & (0.513)\\   
   lnnum\_author                                                              & 19.5$^{***}$\\   
                                     

In [68]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType*ex_ld_max_before_year_with_ih_bin  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_withih <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], vcov = "hetero")
summary(model_pre_withih)

NOTE: 98 observations removed because of NA values (LHS: 98).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 294,781
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                           Estimate Std. Error
CoTypeCollaboration                                       -23.22205   2.301395
CoTypeParticipation                                       -25.83847   3.228196
ex_ld_max_before_year_with_ih_binTrue                     -29.53324   1.012175
lnnum_author                                               21.01265   0.755560
internationalinternational                                  3.36101   0.672651
lnnum_reference                                           -18.09916   1.214429
num_fac                                                   -11.78134   0.289397
SDGTrue                                                    -1.48374   0.552352
lnmean_career_age                                           1.83468   1.028452
lnex_ld_avg_avgimpact                                       7.81367   0.4

In [69]:
# # 每组 reg_class 的平均预测概率
# res_pre_withih <- avg_comparisons(model_pre_withih, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_with_ih_bin')
# res_pre_withih

In [70]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_withih.csv")
# write.csv(res_pre_withih, fname, row.names = FALSE)

In [71]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_withih,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                               & novel\_uzzi\\   
   Model:                                                                            & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                               & -23.2$^{***}$\\   
                                                                                     & (2.30)\\   
   CoTypeParticipation                                                               & -25.8$^{***}$\\   
                                                                                     & (3.23)\\   
   ex\_ld\_max\_before\_year\_with\_ih\_binTrue                                      & -29.5$^{***}$\\   
                                                                                     & (1.01)\\   
   lnnum\_author                                                        

In [72]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType*ex_ld_max_before_year_participation_bin  + ", paper_level, "+", ex_controls, "+", moderating2, "+",disciplines, " | PublishedYear")
)
model_pre_partic <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], vcov = "hetero")
summary(model_pre_partic)

NOTE: 98 observations removed because of NA values (LHS: 98).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 294,781
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                                  Estimate
CoTypeCollaboration                                             -17.143482
CoTypeParticipation                                             -22.907473
ex_ld_max_before_year_participation_binTrue                     -21.923427
lnnum_author                                                     21.155471
internationalinternational                                        3.350200
lnnum_reference                                                 -18.332787
num_fac                                                         -11.466717
SDGTrue                                                          -1.684036
lnmean_career_age                                                 1.053035
lnex_ld_avg_avgimpact                                             7.689534
lnex_ld_avg_insthindex                

In [73]:
# # 每组 reg_class 的平均预测概率
# res_pre_partic <- avg_comparisons(model_pre_partic, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_participation_bin')
# res_pre_partic

In [74]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_partic.csv")
# write.csv(res_pre_partic, fname, row.names = FALSE)

In [75]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_partic,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                                   & novel\_uzzi\\   
   Model:                                                                                & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                                   & -17.1$^{***}$\\   
                                                                                         & (1.28)\\   
   CoTypeParticipation                                                                   & -22.9$^{***}$\\   
                                                                                         & (1.66)\\   
   lnnum\_author                                                                         & 21.2$^{***}$\\   
                                                                                         & (0.760)\\   
   internationalinternational           

In [76]:
fml <- as.formula(
  paste0("novel_uzzi ~ CoType*ex_ld_max_before_year_co_lead_bin  + ", paper_level, "+", ex_controls, "+", moderating3, "+",disciplines, " | PublishedYear")
)
model_pre_co_lead <- feols(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], vcov = "hetero")
summary(model_pre_co_lead)

NOTE: 98 observations removed because of NA values (LHS: 98).



OLS estimation, Dep. Var.: novel_uzzi
Observations: 294,781
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                           Estimate Std. Error
CoTypeCollaboration                                       -20.44579   2.246875
CoTypeParticipation                                         7.50889   2.827249
ex_ld_max_before_year_co_lead_binTrue                     -29.61525   0.918215
lnnum_author                                               20.27970   0.740962
internationalinternational                                  3.53241   0.671168
lnnum_reference                                           -17.99320   1.213442
num_fac                                                   -11.41491   0.285881
SDGTrue                                                    -1.46849   0.552233
lnmean_career_age                                           2.03590   1.028359
lnex_ld_avg_avgimpact                                       8.13985   0.4

In [77]:
# # 每组 reg_class 的平均预测概率
# res_pre_co_lead <- avg_comparisons(model_pre_co_lead, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_co_lead_bin')
# res_pre_co_lead

In [78]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_co_lead.csv")
# write.csv(res_pre_co_lead, fname, row.names = FALSE)

In [79]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_co_lead,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                               & novel\_uzzi\\   
   Model:                                                                            & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                               & -20.4$^{***}$\\   
                                                                                     & (2.25)\\   
   CoTypeParticipation                                                               & 7.51$^{***}$\\   
                                                                                     & (2.83)\\   
   lnnum\_author                                                                     & 20.3$^{***}$\\   
                                                                                     & (0.741)\\   
   internationalinternational                                            

# 补充一个更deep的point，曾经开展过“Co-lead”,后续合作/参与的收益受损更严重

In [88]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", disciplines, " | PublishedYear")
)
model1 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model1)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.168124   0.010601  15.859520
CoTypeParticipation                           0.047151   0.014212   3.317634
Arts.and.Humanities                           1.837644   0.072240  25.438148
Biochemistry..Genetics.and.Molecular.Biology  0.498277   0.012207  40.818163
Business..Management.and.Accounting          -0.428431   0.087633  -4.888910
Chemical.Engineering                          0.131064   0.020722   6.324869
Chemistry                                     0.458689   0.010925  41.986380
Computer.Science                              0.896830   0.039192  22.883139
Decision.Sciences                             1.233745   0.187252   6.588695
Dentistry                                     1.993278   0.134826  14.

In [89]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+",disciplines, " | PublishedYear")
)
model2 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model2)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.176029   0.010963  16.055991
CoTypeParticipation                           0.095087   0.014820   6.416025
lnnum_author                                 -0.130941   0.007103 -18.433963
internationalinternational                   -0.007958   0.008684  -0.916399
lnnum_reference                              -0.005923   0.008397  -0.705350
num_fac                                       0.050942   0.006548   7.779950
SDGTrue                                       0.087651   0.008086  10.840337
lnmean_career_age                            -0.007336   0.012674  -0.578814
Arts.and.Humanities                           1.805187   0.072180  25.009688
Biochemistry..Genetics.and.Molecular.Biology  0.488323   0.012268  39.

In [90]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+",disciplines, " | PublishedYear")
)
model3 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model3)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.143687   0.011097  12.948869
CoTypeParticipation                           0.047011   0.014969   3.140640
lnnum_author                                 -0.068658   0.007116  -9.648419
internationalinternational                   -0.031089   0.009559  -3.252299
lnnum_reference                               0.061520   0.008603   7.150632
num_fac                                       0.059498   0.006605   9.007958
SDGTrue                                       0.092371   0.008114  11.384687
lnmean_career_age                             0.033050   0.012852   2.571617
lnex_ld_avg_avgimpact                        -0.264828   0.007259 -36.483771
lnex_ld_avg_insthindex                       -0.049529   0.007468  -6.

In [91]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model4 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model4)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.099926   0.011245   8.886446
CoTypeParticipation                           0.010299   0.015044   0.684582
lnnum_author                                 -0.087020   0.007328 -11.874570
internationalinternational                   -0.045118   0.009576  -4.711696
lnnum_reference                               0.059057   0.008633   6.841129
num_fac                                       0.062924   0.006715   9.369983
SDGTrue                                       0.093974   0.008128  11.561174
lnmean_career_age                             0.012005   0.013072   0.918354
lnex_ld_avg_avgimpact                        -0.269872   0.007355 -36.690437
lnex_ld_avg_insthindex                       -0.051397   0.007510  -6.

In [92]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(model1, model2, model3, model4,
                    keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + r2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)            & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    CoTypeCollaboration                                 & 0.168$^{***}$  & 0.176$^{***}$  & 0.144$^{***}$  & 0.100$^{***}$\\                                                           & (0.011)        & (0.011)        & (0.011)        & (0.011)\\       CoTypeParticipation                                 & 0.047$^{***}$  & 0.095$^{***}$  & 0.047$^{***}$  & 0.010\\                                                           & (0.014)        & (0.015)        & (0.015)        & (0.015)\\       Arts.and.Humanities                                 & 1.84$^{***}$   & 1.81$^{***}$   & 1.76$^{***}$   & 1.74$^{***}$\\                                                           & (0.072)        & (0.072)        